### Import Dependencies

In [ ]:
import random
from qdrant_client import QdrantClient

import instructor
import json

from pydantic import BaseModel, Field

from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client
import tiktoken

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### Load all data from Qdrant

In [4]:
from qdrant_client.models import Record


HYBRID_SEARCH_COLLECTION_NAME = "Amazon-items-collection-01-hybrid-search"
all_points: list[Record] = []
next_offset = None
batch_size = 100

while True:
  points, next_offset = qdrant_client.scroll(
    collection_name=HYBRID_SEARCH_COLLECTION_NAME,
    limit=batch_size,
    offset=next_offset,
    with_payload=True,
    with_vectors=False
  )
  all_points.extend(points)
  if next_offset is None:
    break

len(all_points)

1000

In [5]:
all_points

[Record(id=1, payload={'preprocessed_description': 'ZGCINE PS-G10 Combo Charging Case with 3 Pcs Batteries for GoPro Hero 10/9, Battery Charger Box for GoPro Hero 10/9/8/7/6/5 Battery, Support USB-C PD Input, with USB-C PD Output and USB-A Output 👍ZGCINE PS-G10 3 Batteries Combo come with 1xZGCINE PS-G10\xa0Charging case, and 3x\xa0ZGCINE Batteries for\xa0GoPro Hero10/9 👍Fully compatible with GoPro original battery: The Charging case is designed for Gopro Hero 10 Hero 9/8/7/6/5 Batteries. With 10400 mAh rechargeable battery, it can fully Charge GoPro hero 10/9 battery in 2.5h, and Fully Charge GoPro hero 8/7/6/5 in 1h50min. 👍Unique Design with Storage Function: Precise slot design, can charge 3pcs GoPro batteries simultaneously. Insert the battery to start charging, easy to operate, just twice press the power button to turn off the charging. It can also store 4 TF Cards with 3 GoPro batteries 👍Reverse Charging Be Power Bank: With USB-C PD output and USB-A out put, it can charge the GoP

### Split data into 2 groups
#### 50 items for synthetic question generation, 950 items to loop through and evaluate their relevance for previously generated synthetic questions

In [6]:
rng = random.Random(42)
indices = list(range(len(all_points)))
indices

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [7]:
rng.shuffle(indices)
indices

[776,
 507,
 895,
 922,
 33,
 483,
 85,
 750,
 354,
 523,
 184,
 809,
 418,
 615,
 682,
 501,
 760,
 49,
 732,
 336,
 450,
 790,
 350,
 467,
 622,
 476,
 614,
 554,
 365,
 770,
 630,
 940,
 797,
 496,
 924,
 369,
 596,
 720,
 53,
 426,
 56,
 264,
 772,
 185,
 466,
 789,
 174,
 150,
 741,
 113,
 428,
 965,
 18,
 189,
 414,
 589,
 666,
 763,
 221,
 586,
 581,
 299,
 638,
 386,
 364,
 879,
 334,
 16,
 202,
 1,
 38,
 290,
 493,
 247,
 506,
 896,
 713,
 891,
 76,
 911,
 792,
 508,
 177,
 842,
 200,
 765,
 909,
 293,
 538,
 24,
 694,
 210,
 424,
 528,
 34,
 928,
 398,
 607,
 298,
 848,
 668,
 282,
 939,
 944,
 606,
 959,
 934,
 731,
 287,
 527,
 807,
 229,
 436,
 28,
 374,
 819,
 316,
 560,
 241,
 149,
 516,
 417,
 660,
 198,
 585,
 325,
 823,
 702,
 949,
 515,
 864,
 836,
 877,
 359,
 594,
 788,
 188,
 66,
 355,
 799,
 902,
 920,
 942,
 330,
 727,
 396,
 463,
 393,
 311,
 680,
 744,
 921,
 461,
 522,
 631,
 490,
 195,
 256,
 693,
 740,
 577,
 157,
 612,
 392,
 969,
 688,
 361,
 816,
 401,
 

In [8]:
sample_idx = set(indices[:50])
sample_idx

{33,
 49,
 53,
 56,
 85,
 113,
 150,
 174,
 184,
 185,
 264,
 336,
 350,
 354,
 365,
 369,
 418,
 426,
 450,
 466,
 467,
 476,
 483,
 496,
 501,
 507,
 523,
 554,
 596,
 614,
 615,
 622,
 630,
 682,
 720,
 732,
 741,
 750,
 760,
 770,
 772,
 776,
 789,
 790,
 797,
 809,
 895,
 922,
 924,
 940}

In [9]:
sample_50 = [p for i, p in enumerate(all_points) if i in sample_idx]

In [10]:
remaining_points = [p for i, p in enumerate(all_points) if i not in sample_idx]

### Generate synthetic questions

In [11]:
from typing import TypedDict
from api.api.models import ItemPayload

Sample = TypedDict("Sample", {"parent_asin": str, "preprocessed_description": str})

all_context_sample_50: list[Sample] = [
    Sample(
        parent_asin=data.parent_asin,
        preprocessed_description=data.preprocessed_description,
    )
    for data in [ItemPayload.model_validate(p.payload) for p in sample_50]
]

In [12]:
all_context_sample_50

[{'parent_asin': 'B0CGR97V51',
  'preprocessed_description': 'Apofial Digital Picture Frame 10.1 Inch WiFi Digital Photo Frame,1280 * 800 HD IPS Touch Screen Smart Cloud Photo Frame, to Share Photos Or Videos Remotely Via APP Email (Black) 【 A Perfect Gift To Keep You Connected】 Instantly share photos from your phone or video to an digital photo frame through free app; friends and family to share pictures to digital picture frame or send photos to their digital picture frames all via WiFi; Additionally, it is a perfect gift for the elderly who are not good at using smart phones to stay connected with the younger generation. 【Impressive features】Withdraw photo via App, send instant or timed wish, sleep mode, show/hide picture, alarm clock mode,Multiple clock templates can be displayed time&weather. single repeat, full screen, adjustable slideshow time ...The digital picture frame offers multiple customized settings to meet your need. 【High Definition Touch Screen】The 10.1" IPS HD touch 

In [13]:
len(all_context_sample_50)

50

In [14]:
all_context_remaining_points: list[Sample] = [
    Sample(
        parent_asin=data.parent_asin,
        preprocessed_description=data.preprocessed_description,
    )
    for data in [ItemPayload.model_validate(p.payload) for p in remaining_points]
]

In [15]:
len(all_context_remaining_points)

950

In [16]:
SYSTEM_PROMPT = f"""
You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant products and grounding its answers in them.

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that will eventually contain 1000 products.

## Your task
Generate exactly 30 questions (MUST BE 30! NO LESS, NO MORE!!!) that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** — answerable using exactly ONE product.
2. **Multi-product (10 questions)** — answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** — plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's point of view, in natural language.
- Refer to the items as "products". Never use the word "chunk".
- Keep questions specific. Even in the full 1000-product collection, a good question should be answerable from at most 4 products. Avoid broad or generic questions that a large number of products could satisfy.

## Products
{json.dumps(all_context_sample_50, indent=2)}
"""

In [17]:
print(SYSTEM_PROMPT)


You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant products and grounding its answers in them.

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that will eventually contain 1000 products.

## Your task
Generate exactly 30 questions (MUST BE 30! NO LESS, NO MORE!!!) that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** — answerable using exactly ONE product.
2. **Multi-product (10 questions)** — answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** — plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's poi

In [18]:
client = instructor.from_provider(
  "openai/gpt-5.4",
  mode=instructor.Mode.RESPONSES_TOOLS
)

In [19]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },
        },
    },
}

class SyntheticQuestion(BaseModel):
  reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
  question: str = Field(description="Suggested question.")
  chunk_ids: list[str] = Field(description="IDs of the chunks that could be used to answer the question.")
  answer_example: str = Field(description="Suggested answer grounded in the context.")

class SyntheticQuestions(BaseModel):
  synthetic_questions: list[SyntheticQuestion] = Field(description="List of synthetic questions", min_length=30, max_length=30)

In [20]:
response, raw_response = client.create_with_completion(
    messages=[{"role": "user", "content": SYSTEM_PROMPT}],
    reasoning={"effort": "none"},
    response_model=SyntheticQuestions,
)

In [21]:
response

SyntheticQuestions(synthetic_questions=[SyntheticQuestion(reasoning='The Apofial digital picture frame description explicitly mentions WiFi sharing via app/email, 10.1-inch 1280x800 IPS touch screen, 16GB built-in storage, unlimited cloud storage, and support for SD card import/export, so one product can answer a question about remote family photo sharing.', question="I'm looking for a digital photo frame I can send pictures to remotely from my phone—does any product have WiFi sharing, a touch screen, and built-in storage?", chunk_ids=['B0CGR97V51'], answer_example='Yes—the Apofial 10.1-inch WiFi digital picture frame supports remote photo and video sharing through a free app or email, has a 1280x800 IPS touch screen, includes 16GB built-in memory, and also supports Micro SD card import/export.'), SyntheticQuestion(reasoning='The Noonkey outdoor camera includes 4MP/2.5K video, 355-degree PTZ coverage, color night vision, two-way audio, siren, custom zones, IP66 waterproofing, and SD/cl

In [22]:
len(response.synthetic_questions)


30

In [23]:
response.synthetic_questions

[SyntheticQuestion(reasoning='The Apofial digital picture frame description explicitly mentions WiFi sharing via app/email, 10.1-inch 1280x800 IPS touch screen, 16GB built-in storage, unlimited cloud storage, and support for SD card import/export, so one product can answer a question about remote family photo sharing.', question="I'm looking for a digital photo frame I can send pictures to remotely from my phone—does any product have WiFi sharing, a touch screen, and built-in storage?", chunk_ids=['B0CGR97V51'], answer_example='Yes—the Apofial 10.1-inch WiFi digital picture frame supports remote photo and video sharing through a free app or email, has a 1280x800 IPS touch screen, includes 16GB built-in memory, and also supports Micro SD card import/export.'),
 SyntheticQuestion(reasoning='The Noonkey outdoor camera includes 4MP/2.5K video, 355-degree PTZ coverage, color night vision, two-way audio, siren, custom zones, IP66 waterproofing, and SD/cloud storage, which fully answers a spe

In [24]:
for s_question in response.synthetic_questions:
  print(s_question.chunk_ids)

['B0CGR97V51']
['B0B18J4J2L']
['B09S5YS68P']
['B0B3MMP22L']
['B0C744R1WC']
['B09QJX5H5T']
['B0C3CTHDPZ']
['B0BLVY5ZFX']
['B0BSRGYNWV']
['B09XDJPBV7']
['B0C1NLT888']
['B0BFX8BCQV']
['B0BY4X57N9']
['B0B59FCVCD']
['B09YKLVMGH']
['B0B3JJGSKP', 'B09TS2F7TW']
['B0B18J4J2L', 'B0BRQDZRXD']
['B09QJX5H5T', 'B0BX32QYBQ']
['B09XDJPBV7', 'B0B18J4J2L']
['B08WX45X6M', 'B09FYH27P7']
['B09PYRVB5T', 'B09V787FQ9']
['B0BL7CKF31', 'B09V2Z7FKD']
['B0C6FDYTJ5', 'B0B59FCVCD']
['B0C1NLT888', 'B0BFX8BCQV']
['B09QJX5H5T', 'B0C1NLT888']
[]
[]
[]
[]
[]


In [25]:
class RelevantData(TypedDict): 
  question_id: int
  question: str

synthetic_questions_relevant_data = [RelevantData(question_id=i, question=item.question) for i, item in enumerate(response.synthetic_questions)]

In [26]:
synthetic_questions_relevant_data

[{'question_id': 0,
  'question': "I'm looking for a digital photo frame I can send pictures to remotely from my phone—does any product have WiFi sharing, a touch screen, and built-in storage?"},
 {'question_id': 1,
  'question': 'Do you have an outdoor security camera that can pan around, record in color at night, and let me talk through it from my phone?'},
 {'question_id': 2,
  'question': 'I need a way to switch between my PS5 and another HDMI device on one TV—do you have a product that supports 4K 120Hz or even 8K?'},
 {'question_id': 3,
  'question': 'Is there a protective case with a screen cover for a Garmin Venu Sq 2 that I can leave on while charging?'},
 {'question_id': 4,
  'question': 'Do you sell Apple-certified Lightning cables in a multipack for iPhone charging?'},
 {'question_id': 5,
  'question': "I'm trying to add wireless Apple CarPlay to an older car without replacing the whole dashboard. Do you have a portable screen for that?"},
 {'question_id': 6,
  'question': 

In [ ]:
SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products
{json.dumps(all_context_remaining_points[0:50], indent=2)}
"""


In [28]:
print(SYSTEM_PROMPT_950)


You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
[
  {
    "question_id": 0,
    "question": "I'm looking for a digital photo frame I can send pictures to remotely from my phone\u2014does any product have WiFi sh

### Count the number of tokens for the static prefix

In [29]:
SYSTEM_PROMPT_STATIC_PART = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products
"""


In [30]:
def count_tokens(text: str) -> int:
  encoding = tiktoken.get_encoding("o200k_base")
  return len(encoding.encode(text))

In [31]:
print(count_tokens(SYSTEM_PROMPT_STATIC_PART))

1375


### Assign chunks to relevant questions for a single batch

In [32]:
client = instructor.from_provider(
  "openai/gpt-5.4-mini",
  mode=instructor.Mode.RESPONSES_TOOLS
)

In [ ]:
class ResultOfMatchingChunksToQuestion(BaseModel):
  reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
  question_id: int = Field(description="ID of the question.")
  chunk_ids: list[str] = Field(description="IDs of the chunks that could be used to answer the question.")

class BatchOfResultsOfMatchingChunksToQuestion(BaseModel):
  results: list[ResultOfMatchingChunksToQuestion] = Field(description="List of results of matching chunks to questions.")

In [34]:
tmp_response, tmp_raw_response = client.create_with_completion(
    messages=[{"role": "user", "content": SYSTEM_PROMPT_950}],
    reasoning={"effort": "none"},
    response_model=BatchOfResultsOfMatchingChunksToQuestion,
)

In [35]:
tmp_response.results

[ResultOfMatchingChunksToQuestion(reasoning='MaxAngel digital picture frame explicitly supports home WiFi, remote sharing via phone app, has a touch screen, and built-in 16GB storage, matching all requested features.', question_id=0, chunk_ids=['B0CHS8T5DH']),
 ResultOfMatchingChunksToQuestion(reasoning='REOLINK Indoor Security Camera is indoor only, so it does not satisfy the outdoor requirement despite having pan/tilt, 2-way audio, and app alerts; no outdoor camera in the set clearly matches all requested features.', question_id=1, chunk_ids=[]),
 ResultOfMatchingChunksToQuestion(reasoning='The portable double din car stereo explicitly supports wireless Apple CarPlay and is a portable screen meant to be mounted without replacing the dashboard; it is the relevant product for adding CarPlay to an older car.', question_id=5, chunk_ids=['B0B421845R']),
 ResultOfMatchingChunksToQuestion(reasoning='Duex Lite is a portable monitor/laptop screen extender and not a car Apple CarPlay screen, w

In [ ]:
for data in tmp_response.results:
  print(f"Question {data.question_id}, Chunks: {data.chunk_ids}")

Question 0, Chunks: ['B0CHS8T5DH']
Question 1, Chunks: []
Question 5, Chunks: ['B0B421845R']
Question 17, Chunks: ['B0B421845R']
Question 18, Chunks: ['B0B8RLKMHC']
Question 19, Chunks: []
Question 20, Chunks: ['B09V787FQ9']
Question 21, Chunks: []
Question 22, Chunks: ['B0B1S22KSW', 'B0CG57QKZ8']
Question 23, Chunks: []
Question 24, Chunks: []
Question 25, Chunks: []
Question 26, Chunks: []
Question 27, Chunks: []
Question 28, Chunks: []
Question 29, Chunks: []
Question 1, Chunks: []
Question 6, Chunks: []
Question 7, Chunks: []
Question 8, Chunks: []
Question 9, Chunks: ['B0B8RLKMHC']
Question 10, Chunks: []
Question 11, Chunks: []
Question 12, Chunks: []
Question 13, Chunks: []
Question 14, Chunks: ['B0B8ZNRS8D']
Question 15, Chunks: ['B0C4H18HHG']
Question 16, Chunks: []
Question 22, Chunks: ['B0B1S22KSW', 'B0CG57QKZ8']
Question 10, Chunks: []


In [38]:
from openai.types.responses import Response


if not isinstance(tmp_raw_response, Response) or tmp_raw_response.usage is None:
        raise ValueError(f"Unexpected raw response: {type(tmp_raw_response)}")
tmp_raw_response.usage

ResponseUsage(input_tokens=19735, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=1603, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=21338)

In [ ]:
def get_relevant_chunks()